In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv


In [2]:
df= pd.read_csv("/kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv")

In [4]:
df.head()

,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-1,Source2,3,2016-02-08 05:46:00,2016-02-08 11:00:00,39.865147,-84.058723,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Night
1,A-2,Source2,2,2016-02-08 06:07:59,2016-02-08 06:37:59,39.928059,-82.831184,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Day
2,A-3,Source2,2,2016-02-08 06:49:27,2016-02-08 07:19:27,39.063148,-84.032608,NaN,NaN,0.01,...,False,False,False,False,True,False,Night,Night,Day,Day
3,A-4,Source2,3,2016-02-08 07:23:34,2016-02-08 07:53:34,39.747753,-84.205582,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Day,Day,Day
4,A-5,Source2,2,2016-02-08 07:39:07,2016-02-08 08:09:07,39.627781,-84.188354,NaN,NaN,0.01,...,False,False,False,False,True,False,Day,Day,Day,Day


In [5]:
df_Ca= df[df['State'] == 'CA']

In [8]:
df_Ca.shape

(7728394, 46)

In [9]:
columns= ['Severity', 'Start_Time', 'Start_Lat','Start_Lng',  'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)','Precipitation(in)', 'Weather_Condition',  'Crossing',  'Junction', 'Railway',  'Stop', 'Traffic_Signal', 'Sunrise_Sunset']

In [10]:

df_Ca=df_Ca[columns]

In [14]:
df_Ca.columns

Index(['Severity', 'Start_Time', 'Start_Lat', 'Start_Lng', 'Temperature(F)',
       'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)',
       'Weather_Condition', 'Crossing', 'Junction', 'Railway', 'Stop',
       'Traffic_Signal', 'Sunrise_Sunset'],
      dtype='object')

In [16]:
df_Ca.notnull().sum()

Severity             1741433
Start_Time           1741433
Start_Lat            1741433
Start_Lng            1741433
Temperature(F)       1741433
Humidity(%)          1741433
Pressure(in)         1741433
Visibility(mi)       1741433
Wind_Speed(mph)      1741433
Weather_Condition    1701655
Crossing             1741433
Junction             1741433
Railway              1741433
Stop                 1741433
Traffic_Signal       1741433
Sunrise_Sunset       1740090
dtype: int64

In [13]:
df_Ca.drop(columns=['Wind_Chill(F)', 'Precipitation(in)'], inplace=True)

In [15]:
df_Ca.fillna({
    'Temperature(F)': df_Ca['Temperature(F)'].mean(),
    'Humidity(%)': df_Ca['Humidity(%)'].mean(),
    'Pressure(in)': df_Ca['Pressure(in)'].mean(),
    'Visibility(mi)': df_Ca['Visibility(mi)'].mean(),
    'Wind_Speed(mph)': df_Ca['Wind_Speed(mph)'].mean()
},inplace=True)

In [17]:
df_Ca.fillna({
    'Weather_Condition': df_Ca['Weather_Condition'].mode()[0],
    'Sunrise_Sunset': df_Ca['Sunrise_Sunset'].mode()[0]
},inplace=True)

In [18]:
df_Ca['Start_Time'] = pd.to_datetime(df_Ca['Start_Time'], errors='coerce')
df_Ca['hour']= df_Ca['Start_Time'].dt.hour
df_Ca['day']= df_Ca['Start_Time'].dt.dayofweek
df_Ca['month']= df_Ca['Start_Time'].dt.month

In [19]:
df_Ca.sample(12)
df_Ca.notnull().sum()
df_Ca.fillna({
    'hour': df_Ca['hour'].mode()[0],
    'day': df_Ca['day'].mode()[0],
    'month': df_Ca['month'].mode()[0]
}, inplace=True)

In [20]:
df_Ca.notnull().sum()
df_Ca.drop(columns= ['Start_Time'], inplace=True)

In [21]:

df_Ca.notnull().sum()

Severity             1741433
Start_Lat            1741433
Start_Lng            1741433
Temperature(F)       1741433
Humidity(%)          1741433
Pressure(in)         1741433
Visibility(mi)       1741433
Wind_Speed(mph)      1741433
Weather_Condition    1741433
Crossing             1741433
Junction             1741433
Railway              1741433
Stop                 1741433
Traffic_Signal       1741433
Sunrise_Sunset       1741433
hour                 1741433
day                  1741433
month                1741433
dtype: int64

In [22]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

ohe= OneHotEncoder(handle_unknown='ignore',sparse_output=False)

In [23]:
cat_cols= ['Weather_Condition',  'Crossing',  'Junction', 'Railway',  'Stop', 'Traffic_Signal', 'Sunrise_Sunset']
preprocessor= ColumnTransformer(transformers=[
    ('cat', ohe, cat_cols)
], remainder= 'passthrough')   
 

In [24]:
x= df_Ca.drop(columns=['Severity'])
y= df_Ca['Severity']


In [25]:
y1=df['Severity']-1

In [28]:
y = (df_Ca['Severity'] >= 3).astype(int)

In [42]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)


# pipeline
model2 = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        objective='binary:logistic',
        n_estimators=300,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_weight=3,
        gamma=0.1,
        reg_lambda=1.0,
        n_jobs=-1,
        random_state=42
    ))
])

# train
model2.fit(x_train, y_train)

# predict (0 or 1)
# y_pred = model2.predict(x_test)
y_prob = model2.predict_proba(x_test)[:, 1]
y_pred = (y_prob > 0.32).astype(int)

# evaluate
print("Accuracy:", model2.score(x_test, y_test))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8868691625010408

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.92      0.93    291390
           1       0.62      0.64      0.63     56897

    accuracy                           0.88    348287
   macro avg       0.77      0.78      0.78    348287
weighted avg       0.88      0.88      0.88    348287


Confusion Matrix:
 [[268709  22681]
 [ 20607  36290]]


In [45]:
import joblib
joblib.dump(model2,"california_model.pkl")

['california_model.pkl']